<a href="https://colab.research.google.com/github/sesay2004/AssignmentRepoDemo/blob/main/Project3_CMSC436.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:

%pip install -q gymnasium

# CMSC 436 - Project 3
# SmartCab Q-Learning Simulation

import gymnasium as gym
import numpy as np
import pandas as pd
import random

# Create SmartCab environment
env = gym.make("Taxi-v4")

# Hyperparameters
alpha_values = [0.01, 0.1, 0.9]
gamma_values = [0.40, 0.99]
epsilon_values = [0.5, 1.0]

# Training settings
total_ep = 15000
total_test_ep = 500
max_steps = 100
# Store results from each experiment
results = []

# Test all 12 hyperparameter combinations
for alpha in alpha_values:
    for gamma in gamma_values:
        for epsilon in epsilon_values:

            print("\n--------------------------------")
            print("Training SmartCab")
            print("Alpha:", alpha)
            print("Gamma:", gamma)
            print("Epsilon:", epsilon)

            # Create a new Q-table for each experiment
            q_table = np.zeros((
                env.observation_space.n,
                env.action_space.n
            ))

            # -----------------------------
            # TRAINING
            # -----------------------------
            for episode in range(total_ep):

                state, info = env.reset()

                for step in range(max_steps):

                    # Exploration or exploitation
                    if random.uniform(0, 1) > epsilon:
                        action = np.argmax(q_table[state, :])
                    else:
                        action = env.action_space.sample()

                    # Take action
                    new_state, reward, terminated, truncated, info = env.step(action)

                    done = terminated or truncated

                    # Update Q-table using Q-Learning
                    if terminated:
                        next_q = 0
                    else:
                        next_q = np.max(q_table[new_state, :])

                    q_table[state, action] = (
                        q_table[state, action]
                        + alpha * (
                            reward
                            + gamma * next_q
                            - q_table[state, action]
                        )
                    )

                    state = new_state

                    if done:
                        break

            print("Training completed!")

            # -----------------------------
            # TESTING
            # -----------------------------
            rewards = []
            penalties_list = []

            for episode in range(total_test_ep):

                state, info = env.reset()

                total_reward = 0
                penalties = 0

                for step in range(max_steps):

                    # Choose the best learned action
                    action = np.argmax(q_table[state, :])

                    new_state, reward, terminated, truncated, info = env.step(action)

                    done = terminated or truncated

                    total_reward += reward

                    # Count illegal pickup/drop-off actions
                    if reward == -10:
                        penalties += 1

                    state = new_state

                    if done:
                        break

                rewards.append(total_reward)
                penalties_list.append(penalties)

            # Calculate average results
            avg_reward = np.mean(rewards)
            avg_penalties = np.mean(penalties_list)

            # Save experiment results
            results.append({
                "Alpha": alpha,
                "Gamma": gamma,
                "Epsilon": epsilon,
                "Average Reward": round(avg_reward, 2),
                "Average Penalties": round(avg_penalties, 2)
            })

            print("Average Reward:", round(avg_reward, 2))
            print("Average Penalties:", round(avg_penalties, 2))

# -----------------------------
# FINAL RESULTS TABLE
# -----------------------------

df = pd.DataFrame(results)

print("\n================================")
print("FINAL RESULTS - QUESTION 1")
print("================================")

print(df.to_string(index=False))

# -----------------------------
# BEST HYPERPARAMETERS
# -----------------------------

best = df.loc[df["Average Reward"].idxmax()]

print("\n================================")
print("BEST HYPERPARAMETER COMBINATION")
print("================================")

print("Alpha:", best["Alpha"])
print("Gamma:", best["Gamma"])
print("Epsilon:", best["Epsilon"])
print("Highest Average Reward:", best["Average Reward"])

# Save results to CSV
df.to_csv("SmartCab_Q_Learning_Results.csv", index=False)

print("\nResults saved to SmartCab_Q_Learning_Results.csv")

env.close()




--------------------------------
Training SmartCab
Alpha: 0.01
Gamma: 0.4
Epsilon: 0.5
Training completed!
Average Reward: -94.31
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.01
Gamma: 0.4
Epsilon: 1.0
Training completed!
Average Reward: -56.01
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.01
Gamma: 0.99
Epsilon: 0.5
Training completed!
Average Reward: -22.79
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.01
Gamma: 0.99
Epsilon: 1.0
Training completed!
Average Reward: -36.8
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.1
Gamma: 0.4
Epsilon: 0.5
Training completed!
Average Reward: 6.17
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.1
Gamma: 0.4
Epsilon: 1.0
Training completed!
Average Reward: 7.87
Average Penalties: 0.0

--------------------------------
Training SmartCab
Alpha: 0.1
Gamma: 0.99
Epsilo